In [1]:
import os
import sys

# 현재 작업 디렉토리 확인해보고
print("CWD:", os.getcwd())

# notebooks/ 에서 한 단계 위(프로젝트 루트)로 올라가기
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
print("PROJECT_ROOT:", PROJECT_ROOT)

# sys.path 에 프로젝트 루트가 없으면 추가
if PROJECT_ROOT not in sys.path:
    sys.path.append(PROJECT_ROOT)
    print("Added to sys.path")


CWD: c:\Users\user\Desktop\project\scalp-vision-agent\notebooks
PROJECT_ROOT: c:\Users\user\Desktop\project\scalp-vision-agent
Added to sys.path


In [2]:
import torch
from torch.utils.data import DataLoader
from torch import optim

from src.config import MASTER_INDEX_CSV
from src.cnn.dataset import ScalpDataset
from src.cnn.models import MultiHeadResNet18
from src.cnn.utils import labels_dict_to_tensor
from src.cnn.losses import multihead_ce_loss
from src.cnn.train import get_device, train_one_epoch, evaluate_one_epoch


In [3]:
device = get_device()
device


device(type='cuda')

In [4]:
BATCH_SIZE = 32
NUM_EPOCHS = 10
LR = 1e-4


In [5]:
from torchvision import transforms as T

# 학습용 transform
train_transform = T.Compose([
    T.Resize((224, 224)),
    T.ToTensor(),
])

# 검증용 transform (지금은 같게 둬도 됨)
val_transform = T.Compose([
    T.Resize((224, 224)),
    T.ToTensor(),
])


In [6]:
from torch.utils.data import Subset

train_dataset = ScalpDataset(index_csv=MASTER_INDEX_CSV, split="train", transforms=train_transform)
val_dataset   = ScalpDataset(index_csv=MASTER_INDEX_CSV, split="val",   transforms=val_transform)

print(len(train_dataset), len(val_dataset))


67588 23568


In [7]:
USE_SUBSET = True   # overfit 테스트: True, 전체 학습: False

if USE_SUBSET:
    subset_indices = list(range(100))  # 앞에서 100장만 사용
    train_dataset_small = Subset(train_dataset, subset_indices)
    train_loader = DataLoader(train_dataset_small, batch_size=BATCH_SIZE, shuffle=True)
else:
    train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)

val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)


In [8]:
model = MultiHeadResNet18()
model.to(device)

optimizer = optim.Adam(model.parameters(), lr=LR)


In [11]:
batch = next(iter(train_loader))

print("type:", type(batch["image"]))
print("shape:", batch["image"].shape)
print("dtype:", batch["image"].dtype)
print("device:", batch["image"].device)


type: <class 'torch.Tensor'>
shape: torch.Size([32, 3, 224, 224])
dtype: torch.float32
device: cpu


In [12]:
for epoch in range(1, NUM_EPOCHS + 1):
    train_loss, train_acc = train_one_epoch(model, train_loader, optimizer, device)
    val_loss, val_acc     = evaluate_one_epoch(model, val_loader, device)

    print(
        f"[{epoch:02d}] "
        f"train_loss={train_loss:.4f}, train_acc={train_acc:.4f} | "
        f"val_loss={val_loss:.4f}, val_acc={val_acc:.4f}"
    )


[01] train_loss=1.1663, train_acc=0.5050 | val_loss=1.5099, val_acc=0.2549
[02] train_loss=0.8951, train_acc=0.7900 | val_loss=1.4148, val_acc=0.3674
[03] train_loss=0.6777, train_acc=0.8933 | val_loss=1.3227, val_acc=0.4809
[04] train_loss=0.5135, train_acc=0.9817 | val_loss=1.2707, val_acc=0.5541
[05] train_loss=0.3881, train_acc=1.0000 | val_loss=1.2504, val_acc=0.5866
[06] train_loss=0.2985, train_acc=1.0000 | val_loss=1.2640, val_acc=0.5954
[07] train_loss=0.2302, train_acc=1.0000 | val_loss=1.2976, val_acc=0.5978
[08] train_loss=0.1812, train_acc=1.0000 | val_loss=1.3370, val_acc=0.5984
[09] train_loss=0.1489, train_acc=1.0000 | val_loss=1.3760, val_acc=0.5986
[10] train_loss=0.1225, train_acc=1.0000 | val_loss=1.4319, val_acc=0.5986
